# 05. Confirmatory tests on new data (Amendment 4)

The main study's RB prediction for M0-M2 was written after M0's result was known. Amendment 4 therefore fixed a
set of predictions **before any of their data existed**, on material and models outside the main study, and the
paper reports them whatever they show. Each prediction is that the RB contrast at $k=4$,
$\Delta_{\mathrm{RB}}(4) = \mathrm{WER(Local)} - \mathrm{WER(Global)}$ with Rubber Band, is positive
(95% interval above zero), unless stated otherwise.

| Test | New material | Prediction |
|---|---|---|
| F1 | M1 and M2 retrained with data-order seeds 2 and 3 | $\Delta_{\mathrm{RB}}(4)>0$ in all six models |
| F2 | three reciters no trained model heard | $\Delta_{\mathrm{RB}}(4)>0$ in M0, M1, M2 |
| F3 | the wav2vec2 CTC recogniser (also the aligner) | $\Delta_{\mathrm{RB}}(4)>0$; growth from $k=2$ to $k=6$ |
| F4 | Whisper-large-v3, not tuned on the Qur'an | gate: clean WER $\le 0.50$; then $\Delta_{\mathrm{RB}}(4)>0$ |
| F5 | English read speech (LibriSpeech test-clean) | $\Delta_{\mathrm{RB}}(4)>0$; RB passes the round-trip check; RB minus PV contrast at $k=2$ positive (the protocol calls this the PV bias) |
| M | existing data | the RB minus PV contrast grows with the PV's artifact imbalance (descriptive) |

In [1]:
import sys, json, gzip, collections
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import numpy as np
import stats as S
import hashlib, re, warnings; warnings.filterwarnings("ignore")
proto = (ROOT / "protocols/PROTOCOL_exposure.md").read_bytes()
print("PROTOCOL_exposure.md sha256:", hashlib.sha256(proto).hexdigest())
print("recorded before the runs:    ", (ROOT / "protocols/PROTOCOL_exposure.sha256").read_text().split()[0])
text = proto.decode()
print(text[text.index("## Amendment 4"):][:1500], "...")
fmt = lambda x: f"{x[0]:+.3f} [{x[1]:+.3f}, {x[2]:+.3f}]"
verdict = lambda x: "SUPPORTED" if x[1] > 0 else "NOT SUPPORTED"
rb4 = S.contrast("nucleus_rb", "global_rb", 4.0)

PROTOCOL_exposure.md sha256: 2ee4e2f30cb461f15cf54e9385189dd6858cca8092905709259b75efcee9a932
recorded before the runs:     2ee4e2f30cb461f15cf54e9385189dd6858cca8092905709259b75efcee9a932
## Amendment 4 (2026-09-21 14:20 Africa/Casablanca). Confirmatory tests on NEW data, fixed BEFORE any of it exists

Motivation: reviewers noted (a) one domain and one model family, (b) the RB prediction E3 was fixed after
M0's RB result was known, (c) one training seed, (d) five confirmatory reciters. Every test below is fixed
before its data are produced or, for the CTC recogniser, before its saved scores are first analysed (the
pipeline has recorded CTC WER in every run since 19 Sep; no CTC contrast has been computed or viewed).
Everything is reported whatever it shows. Primary contrast throughout: Delta_RB(k=4) = WER(RB Local) -
WER(RB Global), equal-reciter/speaker mean, 95% bootstrap interval; "positive" means the interval excludes 0.

F1 Seeds. M1 and M2 are retrained with data-order seeds 2 an

## F2. Reciters no trained model heard
The fresh manifest (`data/manifest_fresh.json`, built by `src/evaluation/make_fresh.py`) draws 60 verse IDs common to the three reciters with numpy seed 20260921. The paper's crossed bootstrap runs with its reciter list set to these three.

In [2]:
F2 = {}
for m in ["tarteel", "M0", "M1", "M2"]:
    d = S.load(f"fresh_{m}", reciters=S.FRESH)
    F2[m] = {"rb4": S.summary(d, rb4, reciters=S.FRESH), "clean": S.summary(d, S.arm("baseline", 1.0), reciters=S.FRESH)}
    per = [np.mean([row[("nucleus_rb", 4.0)] - row[("global_rb", 4.0)] for (n, v), row in d.items() if n == r]) for r in S.FRESH]
    tag = "(descriptive)" if m == "tarteel" else verdict(F2[m]["rb4"])
    print(f"{m:8s} n={len(d)}  RB contrast {fmt(F2[m]['rb4'])}  {tag:13s} per reciter", np.round(per, 3))

tarteel  n=141  RB contrast +0.121 [+0.071, +0.174]  (descriptive) per reciter [0.127 0.098 0.137]


M0       n=141  RB contrast +0.122 [+0.033, +0.214]  SUPPORTED     per reciter [0.12  0.084 0.163]


M1       n=141  RB contrast +0.110 [+0.024, +0.198]  SUPPORTED     per reciter [0.105 0.083 0.141]


M2       n=141  RB contrast +0.135 [+0.038, +0.229]  SUPPORTED     per reciter [0.155 0.058 0.193]


## F1. Retraining seeds
Same recipe, three epochs, same validation subset and selection rule; only the batch order changes. Seed 1 is the original M1/M2. Pre-registered descriptives: the seed spread, and the seed-averaged M2 minus M1 difference against it.

In [3]:
runs = {("M1", 1): "M1", ("M2", 1): "M2", ("M1", 2): "M1_s2", ("M2", 2): "M2_s2", ("M1", 3): "M1_s3", ("M2", 3): "M2_s3"}
F1 = {}
for key, run in runs.items():
    d = S.load(run); F1[key] = (d, S.summary(d, rb4), S.summary(d, S.arm("baseline", 1.0)))
    print(f"{key[0]} seed {key[1]}: RB contrast {fmt(F1[key][1])} {verdict(F1[key][1])}   clean WER {F1[key][2][0]:.3f}")
for v in ("M1", "M2"):
    pts = [F1[(v, s)][1][0] for s in (1, 2, 3)]
    print(f"{v}: seed sd {np.std(pts, ddof=1):.3f}, range {np.ptp(pts):.3f}")
def seed_avg(v, fn):
    acc = collections.defaultdict(list)
    for s in (1, 2, 3):
        for key, row in F1[(v, s)][0].items():
            try: acc[key].append(fn(row))
            except KeyError: pass
    return {k: [float(np.mean(x))] for k, x in acc.items() if len(x) == 3}
for name, fn in [("RB contrast", rb4), ("clean WER", S.arm("baseline", 1.0))]:
    a1, a2 = seed_avg("M1", fn), seed_avg("M2", fn)
    wrap = lambda d: {k: {("x", 0): val[0]} for k, val in d.items()}
    print(f"seed-averaged M2 - M1, {name}: {fmt(S.summary(wrap(a2), lambda r: r[('x', 0)], other=wrap(a1)))}")

M1 seed 1: RB contrast +0.113 [+0.059, +0.178] SUPPORTED   clean WER 0.046


M2 seed 1: RB contrast +0.093 [+0.034, +0.163] SUPPORTED   clean WER 0.025


M1 seed 2: RB contrast +0.106 [+0.034, +0.188] SUPPORTED   clean WER 0.047


M2 seed 2: RB contrast +0.111 [+0.050, +0.181] SUPPORTED   clean WER 0.026


M1 seed 3: RB contrast +0.122 [+0.054, +0.199] SUPPORTED   clean WER 0.054


M2 seed 3: RB contrast +0.131 [+0.072, +0.207] SUPPORTED   clean WER 0.028
M1: seed sd 0.008, range 0.016
M2: seed sd 0.019, range 0.038
seed-averaged M2 - M1, RB contrast: -0.002 [-0.027, +0.027]


seed-averaged M2 - M1, clean WER: -0.023 [-0.053, -0.004]


## F3. A CTC recogniser
The wav2vec2 CTC model also places the target intervals, so it is not independent of them (stated in the paper). Its scores are identical in every run because the audio is identical; they are read from M0's runs.

In [4]:
ctc = S.merged("M0", field="ctc"); ctc2 = S.merged("M2", field="ctc")
print("CTC rows identical between the M0 and M2 runs:", all(ctc[k] == ctc2[k] for k in ctc))
F3 = {"rb4": S.summary(ctc, rb4),
      "growth": S.summary(ctc, lambda r: (r[("nucleus_rb", 6.0)] - r[("global_rb", 6.0)]) - (r[("nucleus_rb", 2.0)] - r[("global_rb", 2.0)])),
      "pv2": S.summary(ctc, S.contrast("nucleus", "global", 2.0))}
print("F3a RB contrast k=4:", fmt(F3["rb4"]), verdict(F3["rb4"]))
print("F3b growth k=2 to 6:", fmt(F3["growth"]), verdict(F3["growth"]))
print("    PV contrast k=2: ", fmt(F3["pv2"]))

CTC rows identical between the M0 and M2 runs: True


F3a RB contrast k=4: +0.126 [+0.065, +0.191] SUPPORTED
F3b growth k=2 to 6: +0.179 [+0.100, +0.262] SUPPORTED
    PV contrast k=2:  -0.056 [-0.133, +0.021]


## F4. Whisper-large-v3
Evaluated from a real-file copy of the Hugging Face files with `config.json` dtype set to float32 (weights identical, upcast); see `results/large_v3/SWAP.json` and the README.

In [5]:
v3 = S.load("large_v3")
F4 = {"clean": S.summary(v3, S.arm("baseline", 1.0)), "rb4": S.summary(v3, rb4), "pv4": S.summary(v3, S.contrast("nucleus", "global", 4.0))}
print("gate, clean WER <= 0.50:", fmt(F4["clean"]), "PASSED" if F4["clean"][0] <= 0.5 else "FAILED")
print("F4 RB contrast k=4:     ", fmt(F4["rb4"]), verdict(F4["rb4"]))
print("   PV contrast k=4:     ", fmt(F4["pv4"]))
print("per reciter RB contrast:", {r: round(np.mean([row[('nucleus_rb', 4.0)] - row[('global_rb', 4.0)] for (n, v), row in v3.items() if n == r]), 3) for r in S.CONFIRMATORY})

gate, clean WER <= 0.50: +0.066 [+0.025, +0.121] PASSED
F4 RB contrast k=4:      +0.044 [-0.002, +0.093] NOT SUPPORTED
   PV contrast k=4:      -0.084 [-0.156, -0.017]
per reciter RB contrast: {'Alafasy': np.float64(0.041), 'Sudais': np.float64(0.069), 'Shuraym': np.float64(0.067), 'Dussary': np.float64(0.018), 'Rifai': np.float64(0.024)}


## F5. English read speech
300 LibriSpeech test-clean utterances (15 from each of 20 speakers), targets = vowels with primary stress of at least 60 ms in forced alignments. Bootstrap: speakers, then utterances within speaker. Round-trip damage is relative to the clean recording for RB and to the unit-rate PV copy for the PV.

In [6]:
F5 = {}
for rec in ["wav2vec2", "whisper_base"]:
    d = S.load_libri(rec); o = {}
    o["clean"] = S.two_stage(d, S.arm("baseline", 1.0))
    for k in (2.0, 4.0, 6.0):
        o[f"rb{k:g}"] = S.two_stage(d, S.contrast("nucleus_rb", "global_rb", k)); o[f"pv{k:g}"] = S.two_stage(d, S.contrast("nucleus", "global", k))
    o["bias2"] = S.two_stage(d, lambda r: S.contrast("nucleus_rb", "global_rb", 2.0)(r) - S.contrast("nucleus", "global", 2.0)(r))
    for a, ref in [("global_rt", "sham_pv_global"), ("nucleus_rt", "sham_pv_local"), ("global_rt_rb", "baseline"), ("nucleus_rt_rb", "baseline")]:
        o[a] = S.two_stage(d, lambda r, a=a, ref=ref: r[(a, 4.0)] - r[(ref, 1.0)])
    F5[rec] = o
    print(f"== {rec}: clean WER {o['clean'][0]:.3f}")
    print("  F5a RB contrast k=4:", fmt(o["rb4"]), verdict(o["rb4"]), "| k=2", fmt(o["rb2"]), "| k=6", fmt(o["rb6"]))
    print("  F5b RB round trips (upper bound < 0.05):", fmt(o["global_rt_rb"]), fmt(o["nucleus_rt_rb"]),
          "PASSED" if max(o["global_rt_rb"][2], o["nucleus_rt_rb"][2]) < 0.05 else "FAILED")
    print("      PV round trips:", fmt(o["global_rt"]), fmt(o["nucleus_rt"]))
    print("  F5c RB minus PV contrast at k=2:", fmt(o["bias2"]), verdict(o["bias2"]), "| PV contrast k=2", fmt(o["pv2"]))

== wav2vec2: clean WER 0.045
  F5a RB contrast k=4: +0.108 [+0.086, +0.130] SUPPORTED | k=2 +0.020 [+0.008, +0.031] | k=6 +0.211 [+0.181, +0.241]
  F5b RB round trips (upper bound < 0.05): +0.006 [+0.000, +0.013] +0.004 [-0.001, +0.009] PASSED
      PV round trips: +0.461 [+0.407, +0.512] +0.010 [+0.002, +0.020]
  F5c RB minus PV contrast at k=2: +0.192 [+0.155, +0.231] SUPPORTED | PV contrast k=2 -0.173 [-0.208, -0.138]


== whisper_base: clean WER 0.077
  F5a RB contrast k=4: +0.074 [+0.049, +0.103] SUPPORTED | k=2 +0.014 [+0.001, +0.027] | k=6 +0.290 [+0.108, +0.615]
  F5b RB round trips (upper bound < 0.05): +0.001 [-0.006, +0.008] +0.007 [+0.000, +0.015] PASSED
      PV round trips: +0.412 [+0.353, +0.481] +0.020 [+0.008, +0.035]
  F5c RB minus PV contrast at k=2: +0.141 [+0.100, +0.183] SUPPORTED | PV contrast k=2 -0.127 [-0.165, -0.090]


## M. Engine difference and the PV's artifact imbalance
Rubber Band is not artifact-free, so the RB minus PV difference is not a direct estimate of PV bias. Across 8 cells (4 models $\times$ $k\in\{2,4\}$): the between-engine difference $\Delta_{\mathrm{RB}}-\Delta_{\mathrm{PV}}$ against the PV's artifact imbalance $A_{\mathrm{PV}}(\mathrm{Global})-A_{\mathrm{PV}}(\mathrm{Local})$.

In [7]:
cells = []
for m in S.MODELS:
    d = S.merged(m)
    for k in (2.0, 4.0):
        bias = S.summary(d, lambda r, k=k: S.contrast("nucleus_rb", "global_rb", k)(r) - S.contrast("nucleus", "global", k)(r))[0]
        imb = S.summary(d, lambda r, k=k: (r[("global_rt", k)] - r[("sham_pv_global", 1.0)]) - (r[("nucleus_rt", k)] - r[("sham_pv_local", 1.0)]))[0]
        cells.append((m, k, imb, bias))
rank = lambda x: np.argsort(np.argsort(x))
x, y = np.array([c[2] for c in cells]), np.array([c[3] for c in cells])
for c in cells: print(f"{c[0]:8s} k={c[1]:g}  imbalance {c[2]:.3f}  RB-PV difference {c[3]:+.3f}")
print("Spearman over 8 cells:", round(float(np.corrcoef(rank(x), rank(y))[0, 1]), 2))
k4 = [c for c in cells if c[1] == 4.0]
print("within k=4:", round(float(np.corrcoef(rank([c[2] for c in k4]), rank([c[3] for c in k4]))[0, 1]), 2))
MECH = (float(np.corrcoef(rank(x), rank(y))[0, 1]), float(np.corrcoef(rank([c[2] for c in k4]), rank([c[3] for c in k4]))[0, 1]))

tarteel  k=2  imbalance 0.451  RB-PV difference +0.110
tarteel  k=4  imbalance 0.285  RB-PV difference +0.005
M0       k=2  imbalance 0.534  RB-PV difference +0.249
M0       k=4  imbalance 0.355  RB-PV difference +0.065
M1       k=2  imbalance 0.482  RB-PV difference +0.207
M1       k=4  imbalance 0.270  RB-PV difference +0.069
M2       k=2  imbalance 0.503  RB-PV difference +0.221
M2       k=4  imbalance 0.291  RB-PV difference +0.008
Spearman over 8 cells: 0.86
within k=4: -0.2


## Check against the paper
Every Amendment 4 number printed in the paper comes from `results/paper_macros/recast_numbers5.tex`. Each is recomputed above and compared here.

In [8]:
paper = dict(re.findall(r"\\newcommand\{\\(\w+)\}\{([^}]*)\}", (ROOT / "results/paper_macros/recast_numbers5.tex").read_text()))
s3, u3 = (lambda v: f"{v:+.3f}"), (lambda v: f"{v:.3f}")
exp = {}
for m, t in {"tarteel": "Tar", "M0": "Mz", "M1": "Mo", "M2": "Mt"}.items():
    exp[f"FrRb{t}"], exp[f"FrRb{t}Lo"], exp[f"FrRb{t}Hi"] = map(s3, F2[m]["rb4"])
own = [F1[(v, s)][1] for v in ("M1", "M2") for s in (1, 2, 3)]
exp["SeedRbMin"], exp["SeedRbMax"] = s3(min(o[0] for o in own)), s3(max(o[0] for o in own))
exp["CtcRb"], exp["CtcRbLo"], exp["CtcRbHi"] = map(s3, F3["rb4"]); exp["CtcGrowth"], exp["CtcGrowthLo"], exp["CtcGrowthHi"] = map(s3, F3["growth"])
exp["VthClean"] = u3(F4["clean"][0]); exp["VthRb"], exp["VthRbLo"], exp["VthRbHi"] = map(s3, F4["rb4"]); exp["VthPv"] = s3(F4["pv4"][0])
for rec, t in [("wav2vec2", "Wv"), ("whisper_base", "Wb")]:
    o = F5[rec]; exp[f"En{t}Clean"] = u3(o["clean"][0])
    exp[f"En{t}Rb"], exp[f"En{t}RbLo"], exp[f"En{t}RbHi"] = map(s3, o["rb4"]); exp[f"En{t}Bias"] = s3(o["bias2"][0])
    exp[f"En{t}RbTwo"], exp[f"En{t}RbSix"], exp[f"En{t}PvTwo"] = s3(o["rb2"][0]), s3(o["rb6"][0]), s3(o["pv2"][0])
    exp[f"En{t}RtPvG"] = u3(o["global_rt"][0])
exp["EnRtRbHiMax"] = u3(max(F5[r][a][2] for r in F5 for a in ("global_rt_rb", "nucleus_rt_rb")))
exp["MechRho"] = f"{MECH[0]:.2f}"
import itertools
split = [np.corrcoef(list(range(8)), list(p) + [4 + i for i in q])[0, 1] for p in itertools.permutations(range(4)) for q in itertools.permutations(range(4))]
exp["MechRhoSplit"] = f"{np.mean(split):.2f}"
for kk, nm in [(2.0, "Two"), (4.0, "Four")]:
    im = [c[2] for c in cells if c[1] == kk]; exp[f"MechImb{nm}Min"], exp[f"MechImb{nm}Max"] = f"{min(im):.2f}", f"{max(im):.2f}"
# post hoc checks printed in the paper: reciter-level t intervals and the boundary-blended local arm
from scipy import stats as st
def tint(x):
    x = np.array(x); se = x.std(ddof=1) / np.sqrt(len(x)); q = st.t.ppf(.975, len(x) - 1); return (x.mean(), x.mean() - q * se, x.mean() + q * se)
per = lambda d, names: [np.mean([row[("nucleus_rb", 4.0)] - row[("global_rb", 4.0)] for (n, v), row in d.items() if n == r]) for r in names]
exp["TintFrMt"], exp["TintFrMtLo"], exp["TintFrMtHi"] = map(s3, tint(per(S.load("fresh_M2", reciters=S.FRESH), S.FRESH)))
exp["TintVth"], exp["TintVthLo"], exp["TintVthHi"] = map(s3, tint(per(v3, S.CONFIRMATORY)))
blend = lambda d: np.mean([np.mean([row[("nucleus_edge", 4.0)] - row[("nucleus", 4.0)] for (n, v), row in d.items() if n == r]) for r in S.CONFIRMATORY])
exp["BlendMax"] = u3(max(abs(blend(S.load(m))) for m in S.MODELS))
print("post hoc t intervals: unseen M2", fmt(tint(per(S.load("fresh_M2", reciters=S.FRESH), S.FRESH))), "| large-v3", fmt(tint(per(v3, S.CONFIRMATORY))))
ok = {k: paper.get(k) == v for k, v in exp.items()}
print(f"{sum(ok.values())} of {len(ok)} checked values match the paper exactly")
for k, v in ok.items():
    if not v: print("MISMATCH", k, "paper:", paper.get(k), "recomputed:", exp[k])

post hoc t intervals: unseen M2 +0.135 [-0.038, +0.308] | large-v3 +0.044 [+0.014, +0.073]
57 of 57 checked values match the paper exactly
